# 01 — Search for a Google Scholar Author via SerpAPI

**SerpAPI** is a web scraping service that provides structured access to search engine results pages. Its `google_scholar_profiles` engine lets you search for authors on Google Scholar without being blocked.

This notebook shows how to:
1. Load the API key from a `.env` file
2. Search for an author by name
3. Inspect the profile results
4. Select the best match and extract the Google Scholar author ID

**Prerequisite:** Create a `.env` file in the repository root (copy `.env.example`) and add your SerpAPI key.  
**SerpAPI docs:** https://serpapi.com/google-scholar-profiles-api

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from serpapi import GoogleSearch

In [2]:
load_dotenv('../.env')  # load SERPAPI_KEY from the .env file in the repo root
SERPAPI_KEY = os.getenv('SERPAPI_KEY')

if not SERPAPI_KEY or SERPAPI_KEY == 'your_serpapi_key_here':
    raise ValueError('SERPAPI_KEY not set. Copy .env.example to .env and add your key.')
print('API key loaded.')

API key loaded.


## 1. Search for a single author by name

In [3]:
author_name = 'Lukas Liss'

params = {
    "engine": "google_scholar",
    "q": f"author:{author_name}",
    "api_key": SERPAPI_KEY,
}

search = GoogleSearch(params)
results = search.get_dict()

profiles = results.get("profiles", {})
authors = profiles.get("authors", [])
print(f'Found {len(authors)} profile(s) for "{author_name}"')

Found 1 profile(s) for "Lukas Liss"


In [4]:
# Inspect the raw result for the first author
if authors:
    print('First profile result:')
    print(json.dumps(authors[0], indent=2))

First profile result:
{
  "name": "Lukas Liss",
  "link": "https://scholar.google.com/citations?user=AK9BE00AAAAJ&hl=en&oi=ao",
  "serpapi_link": "https://serpapi.com/search.json?author_id=AK9BE00AAAAJ&engine=google_scholar_author&hl=en",
  "author_id": "AK9BE00AAAAJ",
  "affiliations": "PhD Student, Process and Data Science, Computer Science, RWTH Aachen University",
  "email": "Verified email at pads.rwth-aachen.de",
  "cited_by": 146
}


## 2. Display all candidate profiles

In [5]:
for i, author in enumerate(authors, 1):
    print(f"{i}. {author.get('name')}")
    print(f"   Author ID   : {author.get('author_id')}")
    print(f"   Affiliation : {author.get('affiliations', 'N/A')}")
    print(f"   Cited by    : {author.get('cited_by', 'N/A')}")
    print(f"   Link        : {author.get('link')}")
    print()

1. Lukas Liss
   Author ID   : AK9BE00AAAAJ
   Affiliation : PhD Student, Process and Data Science, Computer Science, RWTH Aachen University
   Cited by    : 146
   Link        : https://scholar.google.com/citations?user=AK9BE00AAAAJ&hl=en&oi=ao



## 3. Select the best match

In [6]:
if authors:
    best = authors[0]
    author_id = best['author_id']
    print(f'Selected author : {best.get("name")}')
    print(f'Google Scholar ID: {author_id}')
    print()
    print('Use this author_id in notebook 02 to fetch full metrics.')
else:
    print('No profiles found — try a different name spelling.')
    author_id = None

Selected author : Lukas Liss
Google Scholar ID: AK9BE00AAAAJ

Use this author_id in notebook 02 to fetch full metrics.


## 4. Helper function for reuse in later notebooks

In [7]:
def get_scholar_author_id(name: str, api_key: str) -> dict | None:
    """Search Google Scholar for an author by name and return their profile info.
    Returns a dict with author_id, affiliations, cited_by, and link for the first match,
    or None if no profiles found.
    """
    try:
        params = {
            "engine": "google_scholar",
            "q": f"author:{name}",
            "api_key": api_key,
        }
        search = GoogleSearch(params)
        results = search.get_dict()
        profiles = results.get("profiles", {})
        authors = profiles.get("authors", [])
        if authors:
            first = authors[0]
            return {
                "author_id": first.get("author_id"),
                "affiliations": first.get("affiliations"),
                "cited_by": first.get("cited_by"),
                "link": first.get("link"),
            }
        return None
    except Exception as e:
        return None


# Example: look up a second author from the dataset
test_name = 'Lamiaa Abdel-Hamid'
test_result = get_scholar_author_id(test_name, SERPAPI_KEY)
print(f'Result for "{test_name}": {test_result}')

Result for "Lamiaa Abdel-Hamid": {'author_id': 'koRj9FkAAAAJ', 'affiliations': 'Misr International University', 'cited_by': 440, 'link': 'https://scholar.google.com/citations?user=koRj9FkAAAAJ&hl=en&oi=ao'}
